1. Ambiente e configurações
2. Cliente Azure Blob Storage
3. Extração dos estados pela API do IBGE
4. Preparação e validação do DataFrame
5. Persistência local e envio ao Azure Bronze
6. Validação final

In [1]:
from pathlib import Path
from typing import Optional
import datetime as dt
import sys

import pandas as pd
import requests

from IPython.display import display
from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


def detectar_databricks() -> bool:
    try:
        dbutils.secrets
        return True
    except (NameError, AttributeError):
        return False


def localizar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()

    for candidato in [atual, *atual.parents]:
        if (candidato / "src").is_dir():
            return candidato

    raise FileNotFoundError(
        "Pasta 'src' não encontrada no diretório atual "
        "ou nos diretórios pais."
    )


def exigir_valor(
    valor: Optional[str],
    nome: str,
) -> str:
    valor_normalizado = (
        str(valor).strip()
        if valor is not None
        else ""
    )

    if not valor_normalizado:
        raise RuntimeError(
            f"{nome} não configurado."
        )

    return valor_normalizado


def criar_sessao_http() -> requests.Session:
    retry = Retry(
        total=5,
        connect=5,
        read=5,
        status=5,
        backoff_factor=2,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=[
            "GET",
        ],
        raise_on_status=False,
    )

    session = requests.Session()

    session.mount(
        "https://",
        HTTPAdapter(
            max_retries=retry
        ),
    )

    session.headers.update({
        "User-Agent": (
            "Databricks-IBGE-Pipeline/1.0"
        ),
        "Accept": "application/json",
    })

    return session


IS_DATABRICKS = detectar_databricks()
BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(BASE_DIR),
    )

if not IS_DATABRICKS:
    from dotenv import load_dotenv

    ENV_PATH = BASE_DIR / ".env"

    if not ENV_PATH.is_file():
        raise FileNotFoundError(
            f"Arquivo .env não encontrado: "
            f"{ENV_PATH}"
        )

    ambiente_carregado = load_dotenv(
        dotenv_path=ENV_PATH,
        override=True,
    )

    if not ambiente_carregado:
        raise RuntimeError(
            f"Não foi possível carregar: "
            f"{ENV_PATH}"
        )

from src.config.secrets import get_secret

AZURE_STORAGE_ACCOUNT = exigir_valor(
    get_secret(
        "AZURE-STORAGE-ACCOUNT",
        "AZURE_STORAGE_ACCOUNT",
    ),
    "AZURE_STORAGE_ACCOUNT",
)

AZURE_STORAGE_KEY = exigir_valor(
    get_secret(
        "AZURE-STORAGE-KEY",
        "AZURE_STORAGE_KEY",
    ),
    "AZURE_STORAGE_KEY",
)

BRONZE_CONTAINER = exigir_valor(
    get_secret(
        "AZURE-CONTAINER-BRONZE",
        "AZURE_CONTAINER_BRONZE",
    ),
    "AZURE_CONTAINER_BRONZE",
)

LOCAL_BRONZE_PATH = (
    BASE_DIR
    / "dados"
    / "bronze"
)

LOCAL_BRONZE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

azure_credential = AzureNamedKeyCredential(
    name=AZURE_STORAGE_ACCOUNT,
    key=AZURE_STORAGE_KEY,
)

ACCOUNT_URL = (
    f"https://{AZURE_STORAGE_ACCOUNT}"
    ".blob.core.windows.net"
)

blob_service_client = BlobServiceClient(
    account_url=ACCOUNT_URL,
    credential=azure_credential,
)

blob_service_client.get_account_information()

container_client = (
    blob_service_client.get_container_client(
        container=BRONZE_CONTAINER
    )
)

container_client.get_container_properties()

IBGE_ESTADOS_URL = (
    "https://servicodados.ibge.gov.br/"
    "api/v1/localidades/estados"
)

with criar_sessao_http() as session:
    response = session.get(
        url=IBGE_ESTADOS_URL,
        timeout=(30, 120),
    )

    response.raise_for_status()
    estados_json = response.json()

if not isinstance(
    estados_json,
    list,
):
    raise TypeError(
        "A API do IBGE retornou um tipo "
        f"inesperado: "
        f"{type(estados_json).__name__}"
    )

if not estados_json:
    raise ValueError(
        "A API do IBGE retornou uma lista vazia."
    )

df_estados = pd.json_normalize(
    estados_json
)

df_estados = df_estados.rename(
    columns={
        "id": "estado_id",
        "sigla": "estado_sigla",
        "nome": "estado_nome",
        "regiao.id": "regiao_id",
        "regiao.sigla": "regiao_sigla",
        "regiao.nome": "regiao_nome",
    }
)

COLUNAS_OBRIGATORIAS = [
    "estado_id",
    "estado_sigla",
    "estado_nome",
    "regiao_id",
    "regiao_sigla",
    "regiao_nome",
]

colunas_ausentes = [
    coluna
    for coluna in COLUNAS_OBRIGATORIAS
    if coluna not in df_estados.columns
]

if colunas_ausentes:
    raise ValueError(
        f"Colunas ausentes: "
        f"{colunas_ausentes}"
    )

df_estados = df_estados[
    COLUNAS_OBRIGATORIAS
].copy()

df_estados["estado_id"] = (
    pd.to_numeric(
        df_estados["estado_id"],
        errors="raise",
    )
    .astype("int64")
)

df_estados["regiao_id"] = (
    pd.to_numeric(
        df_estados["regiao_id"],
        errors="raise",
    )
    .astype("int64")
)

for coluna in [
    "estado_sigla",
    "estado_nome",
    "regiao_sigla",
    "regiao_nome",
]:
    df_estados[coluna] = (
        df_estados[coluna]
        .astype("string")
        .str.strip()
    )

INGESTED_AT = dt.datetime.now(
    dt.timezone.utc
)

df_estados["_ingested_at"] = (
    INGESTED_AT
)

df_estados["_source"] = (
    IBGE_ESTADOS_URL
)

df_estados = (
    df_estados
    .sort_values(
        by="estado_sigla"
    )
    .reset_index(
        drop=True
    )
)

if df_estados.empty:
    raise lueError(
        "DataFrame IBGE Estados vazio."
    )

colunas_com_nulos = (
    df_estados[
        COLUNAS_OBRIGATORIAS
    ]
    .columns[
        df_estados[
            COLUNAS_OBRIGATORIAS
        ]
        .isnull()
        .any()
    ]
    .tolist()
)

if colunas_com_nulos:
    raise ValueError(
        "Colunas obrigatórias com valores nulos: "
        f"{colunas_com_nulos}"
    )

if df_estados[
    "estado_id"
].duplicated().any():
    ids_duplicados = (
        df_estados.loc[
            df_estados[
                "estado_id"
            ].duplicated(
                keep=False
            ),
            "estado_id",
        ]
        .tolist()
    )

    raise ValueError(
        f"IDs de estado duplicados: "
        f"{ids_duplicados}"
    )

if df_estados[
    "estado_sigla"
].duplicated().any():
    siglas_duplicadas = (
        df_estados.loc[
            df_estados[
                "estado_sigla"
            ].duplicated(
                keep=False
            ),
            "estado_sigla",
        ]
        .tolist()
    )

    raise ValueError(
        f"Siglas de estado duplicadas: "
        f"{siglas_duplicadas}"
    )

DATE_SUFFIX = INGESTED_AT.strftime(
    "%Y-%m-%d"
)

BLOB_NAME = (
    f"{DATE_SUFFIX}_"
    "ibge_estados.parquet"
)

PARQUET_FILE = (
    LOCAL_BRONZE_PATH
    / BLOB_NAME
)

df_estados.to_parquet(
    PARQUET_FILE,
    index=False,
    engine="pyarrow",
)

if not PARQUET_FILE.is_file():
    raise FileNotFoundError(
        f"Parquet não criado: "
        f"{PARQUET_FILE}"
    )

TAMANHO_LOCAL = (
    PARQUET_FILE.stat().st_size
)

if TAMANHO_LOCAL <= 0:
    raise RuntimeError(
        f"Parquet criado sem conteúdo: "
        f"{PARQUET_FILE}"
    )

blob_client = (
    container_client.get_blob_client(
        blob=BLOB_NAME
    )
)

with PARQUET_FILE.open(
    mode="rb"
) as arquivo:
    blob_client.upload_blob(
        data=arquivo,
        overwrite=True,
    )

blob_properties = (
    blob_client.get_blob_properties()
)

if blob_properties.size != TAMANHO_LOCAL:
    raise RuntimeError(
        "O tamanho enviado não corresponde "
        "ao arquivo local. "
        f"Local: {TAMANHO_LOCAL} bytes. "
        f"Azure: {blob_properties.size} bytes."
    )

print(
    "========== RESULTADO =========="
)

print(
    "✅ Ambiente:",
    "Databricks"
    if IS_DATABRICKS
    else "local",
)

print(
    f"✅ Raiz do projeto: "
    f"{BASE_DIR}"
)

print(
    f"✅ Storage Account: "
    f"{AZURE_STORAGE_ACCOUNT}"
)

print(
    f"✅ Container: "
    f"{BRONZE_CONTAINER}"
)

print(
    f"✅ Estados encontrados: "
    f"{len(estados_json)}"
)

print(
    f"✅ Estados únicos: "
    f"{df_estados['estado_sigla'].nunique()}"
)

print(
    f"✅ Regiões únicas: "
    f"{df_estados['regiao_sigla'].nunique()}"
)

print(
    f"✅ Arquivo local: "
    f"{PARQUET_FILE.resolve()}"
)

print(
    f"✅ Blob Azure: "
    f"{BLOB_NAME}"
)

print(
    f"✅ Tamanho local: "
    f"{TAMANHO_LOCAL / 1024:.2f} KB"
)

print(
    f"✅ Tamanho Azure: "
    f"{blob_properties.size / 1024:.2f} KB"
)

print(
    f"✅ Ingestão UTC: "
    f"{INGESTED_AT.isoformat()}"
)

print(
    "✅ Parquet local e blob Azure "
    "validados com sucesso."
)

display(
    df_estados
    if IS_DATABRICKS
    else df_estados.head(10)
)

========== RESULTADO ==========
✅ Ambiente: local
✅ Raiz do projeto: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28
✅ Storage Account: stfiaptechchallenge2iast
✅ Container: bronze
✅ Estados encontrados: 27
✅ Estados únicos: 27
✅ Regiões únicas: 5
✅ Arquivo local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-18_ibge_estados.parquet
✅ Blob Azure: 2026-10-18_ibge_estados.parquet
✅ Tamanho local: 5.99 KB
✅ Tamanho Azure: 5.99 KB
✅ Ingestão UTC: 2026-10-18T13:04:16.902489+00:00
✅ Parquet local e blob Azure validados com sucesso.


,estado_id,estado_sigla,estado_nome,regiao_id,regiao_sigla,regiao_nome,_ingested_at,_source
0,12,AC,Acre,1,N,Norte,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
1,27,AL,Alagoas,2,NE,Nordeste,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
2,13,AM,Amazonas,1,N,Norte,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
3,16,AP,Amapá,1,N,Norte,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
4,29,BA,Bahia,2,NE,Nordeste,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
5,23,CE,Ceará,2,NE,Nordeste,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
6,53,DF,Distrito Federal,5,CO,Centro-Oeste,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
7,32,ES,Espírito Santo,3,SE,Sudeste,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
8,52,GO,Goiás,5,CO,Centro-Oeste,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
9,21,MA,Maranhão,2,NE,Nordeste,2026-10-18 13:04:16.902489+00:00,https://servicodados.ibge.gov.br/api/v1/locali...
